# Workflow for Preprocessing Individual Datasets

# Load and Read Visium Data
Visium files for each dataset were downloaded and arranged in the manner of CellRanger outputs for consistency in reading individual samples.

In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
def map_sample_to_patient(sample_name, patient_df):    
    base_name = sample_name.split('-')[0]
    
    patient_info = patient_df[patient_df['folder_name'] == base_name]
    
    if len(patient_info) > 0:
        return patient_info.iloc[0]
    else:
        print(f"No patient metadata found for {sample_name}")
        return None

In [ ]:
def read_visium_sample(sample_name, base_dir, patient_df=None):    
    sample_path = os.path.join(base_dir, sample_name)
    
    matrix_dir = os.path.join(sample_path, "filtered_feature_bc_matrix")
    spatial_dir = os.path.join(sample_path, "spatial")
    
    h5_filename = None
    if os.path.exists(matrix_dir):
        h5_files = [f for f in os.listdir(matrix_dir) if f.endswith('.h5')]
        if h5_files:
            h5_filename = h5_files[0]    
    try:
        counts_file_path = f"filtered_feature_bc_matrix/{h5_filename}"
        
        adata = sq.read.visium(
            path=sample_path,
            counts_file=counts_file_path,
            library_id=sample_name
        )
        
        adata.obs['sample'] = sample_name
        
        if patient_df is not None:
            patient_info = map_sample_to_patient(sample_name, patient_df)
        
        print(f"Shape: {adata.shape}")
        
        return adata
        
    except Exception as e:
        print(f"squidpy.read.visium failed: {str(e)}")
        print(f"gotta try manual loading")
        
        try:
            return manual_load_spatial_data(sample_path, sample_name, h5_filename, patient_df)
        except Exception as e2:
            print(f"   manual loading also failed: {str(e2)}")
            return None



In [ ]:
# if squidpy loading doesnt work:
def manual_load_spatial_data(sample_path, sample_name, h5_filename, patient_df=None):
    
    matrix_dir = os.path.join(sample_path, "filtered_feature_bc_matrix")
    spatial_dir = os.path.join(sample_path, "spatial")
    h5_path = os.path.join(matrix_dir, h5_filename)
    
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()
    
    # Try different tissue position file names
    tissue_positions_files = [
        "tissue_positions_list.csv",  # Standard Visium
        "tissue_positions.csv"  # CytAssist format
    ]
    
    positions_loaded = False
    for positions_file in tissue_positions_files:
        tissue_positions_path = os.path.join(spatial_dir, positions_file)
        if os.path.exists(tissue_positions_path):
            
            with open(tissue_positions_path, 'r') as f:
                first_line = f.readline()
                has_header = 'barcode' in first_line.lower()
            
            if has_header:
                positions = pd.read_csv(tissue_positions_path, index_col=0)
            else:
                positions = pd.read_csv(tissue_positions_path, header=None, index_col=0)
                positions.columns = ['in_tissue', 'array_row', 'array_col', 
                                   'pxl_row_in_fullres', 'pxl_col_in_fullres']
            
            common_barcodes = adata.obs.index.intersection(positions.index)
            positions_matched = positions.loc[common_barcodes]
            
            # Add spatial info to adata
            adata.obs['in_tissue'] = positions_matched['in_tissue'].astype(bool)
            adata.obs['array_row'] = positions_matched['array_row']
            adata.obs['array_col'] = positions_matched['array_col']
            adata.obsm['spatial'] = positions_matched[['pxl_row_in_fullres', 
                                                       'pxl_col_in_fullres']].values
            
            positions_loaded = True
            break
    
    
    # Read scale factors
    scalefactors_file = os.path.join(spatial_dir, "scalefactors_json.json")
    if os.path.exists(scalefactors_file):
        with open(scalefactors_file, 'r') as f:
            scalefactors = json.load(f)
        adata.uns['spatial'] = {sample_name: {'scalefactors': scalefactors}}
    
    # Add images if available
    for img_name in ['tissue_hires_image.png', 'tissue_lowres_image.png']:
        img_path = os.path.join(spatial_dir, img_name)
        if os.path.exists(img_path):
            img = plt.imread(img_path)
            if 'spatial' not in adata.uns:
                adata.uns['spatial'] = {}
            if sample_name not in adata.uns['spatial']:
                adata.uns['spatial'][sample_name] = {}
            if 'images' not in adata.uns['spatial'][sample_name]:
                adata.uns['spatial'][sample_name]['images'] = {}
            
            img_key = 'hires' if 'hires' in img_name else 'lowres'
            adata.uns['spatial'][sample_name]['images'][img_key] = img
    
    # Add sample name
    adata.obs['sample'] = sample_name
    
    # Add clinical metadata if available
    if patient_df is not None:
        patient_info = map_sample_to_patient(sample_name, patient_df)
        if patient_info is not None:
            add_clinical_metadata_to_adata(adata, patient_info, sample_name)
    
    print(f"   Shape: {adata.shape}")
    
    return adata



In [ ]:
# use supplemental table or whatever provided clinical data from paper to read in
# assuming that clinical data is in patient_info 
def add_clinical_metadata_to_adata(adata, patient_info, sample_name):
    adata.obs['patient_id'] = patient_info['PatientID']
    adata.obs['age'] = patient_info['Age']
    adata.obs['gender'] = patient_info['Gender']
    adata.obs['tissue_type'] = patient_info['Tissue']
    adata.obs['tumor_volume'] = patient_info['Tumor_volume_cm3']
    adata.obs['cytassist'] = patient_info['CytAssist']
    adata.obs['visium_ffpe'] = patient_info['10x_Visium_FFPE_ST']
    
    
def batch_load_all_samples(sample_dirs, base_dir, patient_df):
    """Load all samples with metadata"""
        
    all_adata = []
    failed_samples = []
    loading_summary = []
    
    for i, sample in enumerate(sample_dirs, 1):        
        try:
            adata = read_visium_sample(sample, base_dir, patient_df)
            
            if adata is not None:
                all_adata.append(adata)
                
                summary_info = {
                    'sample_id': sample,
                    'patient_id': adata.obs['patient_id'].iloc[0] if 'patient_id' in adata.obs else None,
                    'tissue_type': adata.obs['tissue_type'].iloc[0] if 'tissue_type' in adata.obs else None,
                    'n_spots_total': adata.shape[0],
                    'n_spots_in_tissue': adata.obs['in_tissue'].sum() if 'in_tissue' in adata.obs else adata.shape[0],
                    'n_genes': adata.shape[1],
                    'age': adata.obs['age'].iloc[0] if 'age' in adata.obs else None,
                    'gender': adata.obs['gender'].iloc[0] if 'gender' in adata.obs else None,
                    'clinical_stage': adata.obs['clinical_stage'].iloc[0] if 'clinical_stage' in adata.obs else None,
                    'platform': 'CytAssist' if adata.obs['cytassist'].iloc[0] == 'Yes' else 'Visium FFPE'
                }
                loading_summary.append(summary_info)
                
            else:
                failed_samples.append(sample)
                
        except Exception as e:
            failed_samples.append(sample)
    
    # Create summary DataFrame
    summary_df = pd.DataFrame(loading_summary)
    
    
    return all_adata, summary_df, failed_samples


# Save individual H5ad files 

In [ ]:
def save_individual_samples_as_h5ad(all_adata, output_dir="output"):

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
        
    saved_files = []
    save_summary = []
    failed_saves = []
    
    
    for i, adata in enumerate(all_adata, 1):
        try:
            sample_id = adata.obs['sample'].iloc[0] if 'sample' in adata.obs.columns else f"sample_{i}"
            patient_id = adata.obs.get('patient_id', pd.Series([f"unknown_patient_{i}"])).iloc[0]
            platform = 'CytAssist' if adata.obs['cytassist'].iloc[0] == 'Yes' else 'VisiumFFPE'
            clinical_stage = adata.obs.get('clinical_stage', pd.Series(['unknown'])).iloc[0]
            
            patient_clean = patient_id.replace('#', '').replace(' ', '_')
            filename = f"{sample_id}_{patient_clean}_{platform}_{clinical_stage}_{ni_group}.h5ad"
            filepath = output_path / filename
            
            has_spatial_coords = 'spatial' in adata.obsm
            has_spatial_metadata = 'spatial' in adata.uns
            has_tissue_info = 'in_tissue' in adata.obs
            
            if has_spatial_coords:
                spatial_shape = adata.obsm['spatial'].shape
                
            if has_tissue_info:
                n_in_tissue = adata.obs['in_tissue'].sum()
            
            adata.write(filepath, compression='gzip')
            
            save_info = {
                'sample_id': sample_id,
                'patient_id': patient_id,
                'platform': platform,
                'clinical_stage': clinical_stage,
                'filename': filename,
                'filepath': str(filepath),
                'n_spots': adata.shape[0],
                'n_genes': adata.shape[1],
                'n_spots_in_tissue': adata.obs['in_tissue'].sum() if 'in_tissue' in adata.obs else adata.shape[0],
                'has_spatial_coords': has_spatial_coords,
                'has_spatial_metadata': has_spatial_metadata,
                'file_size_mb': filepath.stat().st_size / (1024*1024) if filepath.exists() else 0
            }
            save_summary.append(save_info)
            saved_files.append(str(filepath))
                        
        except Exception as e:
            failed_saves.append({
                'sample_id': sample_id if 'sample_id' in locals() else f"sample_{i}",
                'error': str(e)
            })
    
    summary_df = pd.DataFrame(save_summary)
    
    if failed_saves:
        print(f"Failed saves:")
        for fail in failed_saves:
            print(f"  - {fail['sample_id']}: {fail['error']}")
    
    if len(saved_files) > 0:
        total_size_mb = summary_df['file_size_mb'].sum()
        avg_size_mb = summary_df['file_size_mb'].mean()
                
        summary_file = output_path / "save_summary.csv"
        summary_df.to_csv(summary_file, index=False)
        print(f"   Summary table saved: {summary_file}")
        
        
    return saved_files, summary_df, failed_saves

saved_files, summary_df, failed_saves = save_individual_samples_as_h5ad(
    all_adata, 
    output_dir="output"
)

# Concatenate and Pre-process Anndatas

In [ ]:
# all_adata should be a list of anndatas that were individually loaded

def concatenate_spatial_data(all_adata, output_dir, create_normalized=True):
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    prepared_adata = []
    keys = []
    
    for i, adata in enumerate(all_adata):
        print(f"Processing sample {i+1}/{len(all_adata)}: {adata.obs['sample'].iloc[0]}")
        
        adata_copy = adata.copy()
        
        if sp.issparse(adata_copy.X):
            adata_copy.X = adata_copy.X.tocsr()
        
        adata_copy.var_names_make_unique()
        
        if 'n_genes_by_counts' not in adata_copy.obs.columns:
            print(f"   Calculating QC metrics...")
            sc.pp.calculate_qc_metrics(adata_copy, inplace=True)
        
        sample_key = adata_copy.obs['sample'].iloc[0]
        keys.append(sample_key)
        
        prepared_adata.append(adata_copy)
        print(f"   Shape: {adata_copy.shape}, Key: {sample_key}")
    
    # create combined dataset (RAW COUNTS - for cell2location)    
    adata_all_raw = sc.concat(
        prepared_adata,
        label="library_id",
        keys=keys,
        index_unique="-",
        uns_merge="unique",
        axis=0
    )
    
    for col in adata_all_raw.obs.columns:
        if col in ['in_tissue', 'array_row', 'array_col']:
            adata_all_raw.obs[col] = pd.to_numeric(adata_all_raw.obs[col], errors='coerce')
        elif col in ['age', 'mIHC']:
            adata_all_raw.obs[col] = pd.to_numeric(adata_all_raw.obs[col], errors='coerce')
    
    if "spatial" in adata_all_raw.obsm:
        adata_all_raw.obsm["spatial"] = np.array(adata_all_raw.obsm["spatial"], dtype=np.float32)
    
    output_file_all = os.path.join(output_dir, "spatial_all_raw.h5ad")
    adata_all_raw.write(output_file_all)

    
    if create_normalized:
        
        # Normalize all samples
        adata_all_norm = adata_all_raw.copy()
        print("   Normalizing all samples...")
        sc.pp.normalize_total(adata_all_norm, target_sum=1e4)
        sc.pp.log1p(adata_all_norm)
        
        # Find highly variable genes
        sc.pp.highly_variable_genes(adata_all_norm, flavor="seurat", n_top_genes=3000, 
                                  batch_key="library_id", inplace=True)
        
        output_file_all_norm = os.path.join(output_dir, "GSE278694_spatial_all_normalized.h5ad")
        adata_all_norm.write(output_file_all_norm)
                
        print(f"   ✅ Saved normalized versions")
        
        return adata_all_raw, adata_all_norm
    
    else:
        return adata_all_raw